<div dir="rtl" style="text-align:right">
<h1 style="text-align:right">یک فایل سالم، یک معنی ثابت</h1>
<p style="text-align:right">درس 57 از 92 · چه چیزی باید همراه وزن‌ها ذخیره شود؟ · <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">50-checkpoint</code></p>
<p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-02/50-checkpoint.html">📖 بازگشت به همین درس</a></p>
<p style="text-align:right">بازیابی مدل را با وزن‌ها و ترتیب <bdi dir="ltr">Vocabulary</bdi> بیازمایید، نه فقط با بازشدن فایل.</p><p style="text-align:right">پیش‌نیاز: <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state_dict</code>، <bdi dir="ltr">Tokenizer</bdi> و کپی مستقل <bdi dir="ltr">Tensor</bdi>.</p>
<p style="text-align:right">زمان یادگیری درس همراه با همین دفتر: حدود ۹۰–۱۴۵ دقیقه. زمان دفتر دوباره به زمان درس اضافه نمی‌شود؛ نصب و تمرین اختیاری جداست.</p>
<p style="text-align:right">این دفتر نیمهٔ عملی درس است. مثال‌ها آمادهٔ اجرا هستند؛ دو <bdi dir="ltr">Cell</bdi> با برچسب <bdi dir="ltr">TODO</bdi> را خودتان کامل کنید. پیام <bdi dir="ltr">INCOMPLETE</bdi> یعنی هنوز چیزی ننوشته‌اید، نه اینکه پاسخ درست است. جواب مرجع در این دفتر پنهان نشده است.</p>
<p style="text-align:right">از بالا به پایین اجرا کنید. پس از تغییر هر تابع، <bdi dir="ltr">Cell</bdi> آن و سپس <bdi dir="ltr">Cell</bdi> آزمون را دوباره اجرا کنید. برای بررسی نهایی، از منوی <code style="direction:ltr;text-align:left;unicode-bidi:isolate">Kernel → Restart Kernel and Run All Cells</code> استفاده کنید.</p>
</div>

In [ ]:
from pathlib import Path
import os
import sys

project_root = next((p for p in (Path.cwd(), *Path.cwd().parents)
                     if (p / "mini_gpt").is_dir() and (p / "book_src").is_dir()), None)
if project_root is None:
    raise RuntimeError("Extract the complete learning project; open this notebook inside it.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
print("Python:", sys.executable)
print("Project:", project_root)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">قبل از اجرا، پیش‌بینی کنید</h2>
<p style="text-align:right">اگر دو کاراکتر در <bdi dir="ltr">Vocabulary</bdi> جابه‌جا شوند ولی شکل همهٔ وزن‌ها درست باشد، کدام آزمون باید شکست بخورد؟</p>
</div>

<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی من: …</p></div>

In [ ]:
import copy
import tempfile
from pathlib import Path
import torch
from mini_gpt.config import ModelConfig
from mini_gpt.model import MiniGPT
from mini_gpt.tokenizer import CharacterTokenizer
from mini_gpt.checkpoint import save_checkpoint,load_checkpoint
torch.set_num_threads(1)
torch.manual_seed(7)
tokenizer = CharacterTokenizer.from_text('abc')
model = MiniGPT(ModelConfig(tokenizer.vocab_size,4,8,2,1,0.0))
optimizer = torch.optim.AdamW(model.parameters(),lr=0.001)
x,y = torch.tensor([[1,2,3]]),torch.tensor([[2,3,1]])
loss = model(x,y)[1]
loss.backward()
optimizer.step()
with tempfile.TemporaryDirectory(prefix='aibook-checkpoint-') as directory:
    path = Path(directory)/'example.pt'
    save_checkpoint(path,model,tokenizer,optimizer,1,{},torch.Generator().manual_seed(8),loss.item())
    restored,restored_tokenizer,payload = load_checkpoint(path)
print('restored step:',payload['step'],'optimizer has history:',bool(payload['optimizer']['state']))

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">این بار شما کد بنویسید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">same_artifact(left, right, left_tokens, right_tokens)</code> دو <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state_dict</code> و دو فهرست <bdi dir="ltr">Vocabulary</bdi> را مقایسه کند. فقط اگر کلیدها، مقدار <bdi dir="ltr">Tensor</bdi>ها و ترتیب <bdi dir="ltr">Vocabulary</bdi> یکسان‌اند <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">True</code> بدهد.</p>
</div>

In [ ]:
def same_artifact(left, right, left_tokens, right_tokens):
    # TODO: شکل درست به‌تنهایی کافی نیست
    return None

In [ ]:
def test_exercise():
    a,b = model.state_dict(),restored.state_dict()
    tokens = tokenizer.id_to_token
    result = same_artifact(a,b,tokens,restored_tokenizer.id_to_token)
    if result is None:
        return False
    assert result is True
    swapped = list(tokens)
    swapped[1],swapped[2] = swapped[2],swapped[1]
    assert same_artifact(a,b,tokens,swapped) is False
    changed = {key:value.clone() for key,value in b.items()}
    changed['token_embedding.weight'][1,0] += 1
    assert same_artifact(a,changed,tokens,tokens) is False
    assert same_artifact(a,{},tokens,tokens) is False
    return True
exercise_complete = test_exercise()
print('PASS' if exercise_complete else 'INCOMPLETE: same_artifact')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">فقط یک عامل را تغییر دهید</h2>
<p style="text-align:right">فقط یک وزنِ مدل بازیابی‌شده را عوض کنید و اختلاف <bdi dir="ltr">Logits</bdi> ورودی ثابت را ببینید؛ مدل اصلی دست‌نخورده بماند.</p>
</div>

In [ ]:
trial = copy.deepcopy(restored).eval()
model.eval()
with torch.no_grad():
    baseline = model(x)[0]
    trial.language_model_head.weight[1,0] += 0.5
    print('maximum logit difference:',(trial(x)[0]-baseline).abs().max().item())

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">خرابی را پیدا کنید</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">state_dict</code> به‌تنهایی کپی مستقلی از وزن‌های در حال تغییر نیست. <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">snapshot(model)</code> را بنویسید تا همهٔ <bdi dir="ltr">Tensor</bdi>ها <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">detach</code> و <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">clone</code> شوند.</p>
</div>

In [ ]:
trial = copy.deepcopy(model)
wrong_snapshot = trial.state_dict()
old_value = wrong_snapshot['token_embedding.weight'][1,0].item()
with torch.no_grad():
    trial.token_embedding.weight[1,0] += 1
print('snapshot changed too:',wrong_snapshot['token_embedding.weight'][1,0].item()!=old_value)

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">اصلاح را خودتان بنویسید</h2>
<p style="text-align:right">علت را توضیح دهید، سپس تابع زیر را کامل کنید. خطای عمدی بالا یک نمونهٔ آموزشی است؛ آزمون پایین باید اصلاح شما را بسنجد.</p>
</div>

In [ ]:
def snapshot(model):
    # TODO: نگاشت مستقل از تغییرهای بعدی مدل
    return None

In [ ]:
def test_repair():
    trial = copy.deepcopy(model)
    result = snapshot(trial)
    if result is None:
        return False
    old = result['token_embedding.weight'].clone()
    with torch.no_grad():
        trial.token_embedding.weight.add_(1)
    assert torch.equal(old,result['token_embedding.weight'])
    assert result.keys()==trial.state_dict().keys()
    assert all(not value.requires_grad for value in result.values())
    return True
repair_complete = test_repair()
print('PASS' if repair_complete else 'INCOMPLETE: snapshot')

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">در <bdi dir="ltr">Mini-GPT</bdi> کجا به کار می‌آید؟</h2>
<p style="text-align:right"><code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">save_checkpoint</code>/<code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">load_checkpoint</code> واقعی استفاده شدند. <bdi dir="ltr">Metadata</bdi> خالی این مثال برای نمایش بازیابی است؛ برای <bdi dir="ltr">Resume</bdi> کامل، فایل را با <code dir="ltr" style="direction:ltr;text-align:left;unicode-bidi:isolate">train.py</code> بسازید. فایل موقت در پایان <bdi dir="ltr">setup</bdi> پاک شده است.</p>
</div>

<div dir="rtl" style="text-align:right">
<h2 style="text-align:right">با زبان خودتان توضیح دهید</h2>
<p style="text-align:right">چرا هم برابری <bdi dir="ltr">Tensor</bdi>ها و هم برابری <bdi dir="ltr">Vocabulary</bdi> لازم بود؟</p>
</div>
<div dir="rtl" style="text-align:right"><p style="text-align:right">پیش‌بینی و مشاهدهٔ من: …</p><p style="text-align:right">علت خرابی و اصلاح من: …</p></div>

<div dir="rtl" style="text-align:right"><p style="text-align:right"><a target="_self" href="http://127.0.0.1:8000/part-08/chapter-02/50-checkpoint.html">بازگشت به درس و ادامهٔ مسیر</a> · <a target="_self" href="http://127.0.0.1:8000/answers/50-checkpoint.html#lab-solution">فقط پس از تلاش: راه‌حل مرجع آزمایشگاه</a></p></div>